# smishfuse-full-c2

Generated by `scripts/make_kernels.py` (SmishFuse). Spec:

```json
{
 "kind": "neural",
 "mode": "full",
 "budget_hours": 8.5,
 "state_from": [
  "smishfuse-full-llm-bert",
  "smishfuse-full-bert-distil"
 ],
 "queues": [
  [
   {
    "task": "mendeley_3class",
    "model": "qwen_fewshot",
    "check_cache": true
   },
   {
    "task": "mendeley_3class",
    "model": "qwen_fewshot"
   }
  ],
  [
   {
    "task": "super_sms_clean_binary",
    "model": "bert"
   },
   {
    "task": "moz_binary",
    "model": "qwen_fewshot",
    "check_cache": true
   },
   {
    "task": "moz_binary",
    "model": "qwen_fewshot"
   }
  ]
 ]
}
```


In [ ]:
import glob, json, os, shutil, subprocess, sys, tarfile, zipfile
KERNEL = 'smishfuse-full-c2'
SPEC = json.loads('{"kind": "neural", "mode": "full", "budget_hours": 8.5, "state_from": ["smishfuse-full-llm-bert", "smishfuse-full-bert-distil"], "queues": [[{"task": "mendeley_3class", "model": "qwen_fewshot", "check_cache": true}, {"task": "mendeley_3class", "model": "qwen_fewshot"}], [{"task": "super_sms_clean_binary", "model": "bert"}, {"task": "moz_binary", "model": "qwen_fewshot", "check_cache": true}, {"task": "moz_binary", "model": "qwen_fewshot"}]]}')
PIP = ['optuna', 'bitsandbytes', 'accelerate']

def find_code():
    hits = sorted(glob.glob("/kaggle/input/**/configs/step3.yaml", recursive=True))
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    # The code dataset may arrive as archives: unpack them and search again.
    tmp = "/tmp/smishfuse_code"
    for a in glob.glob("/kaggle/input/**/smishfuse-code/**/*", recursive=True):
        if a.endswith(".zip"):
            zipfile.ZipFile(a).extractall(os.path.join(tmp, os.path.basename(a)[:-4]))
        elif a.endswith((".tar", ".tar.gz", ".tgz")):
            tarfile.open(a).extractall(tmp)
    hits = sorted(glob.glob(tmp + "/**/configs/step3.yaml", recursive=True))
    if not hits:
        raise SystemExit("smishfuse-code not found")
    return os.path.dirname(os.path.dirname(hits[0]))

code = find_code()
shutil.copytree(code, "/kaggle/working/code", dirs_exist_ok=True)
print("code from", code)
if PIP:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP], check=True)


In [ ]:
import subprocess, sys
p = subprocess.Popen([sys.executable, "/kaggle/working/code/src/kaggle_entry.py", "--kernel", KERNEL,
                      "--spec", json.dumps(SPEC)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
rc = p.wait()
print("exit code", rc)
if rc:
    raise SystemExit(rc)
